In [1]:
# ============================================================
# EXPERIMENT 002
# Logistic-regression linear control
# ============================================================

from pathlib import Path
from time import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


warnings.filterwarnings("ignore", category=ConvergenceWarning)


# ============================================================
# 1. CONFIGURATION
# ============================================================

EXPERIMENT_ID = "EXP-002"

RANDOM_STATE = 42
N_SPLITS = 3

TARGET = "addicted_label"
ID_COLUMN = "id"

# Only prepare a Kaggle submission if the linear model is
# competitive enough to give us useful leaderboard evidence.
SUBMISSION_THRESHOLD = 0.960


# Make paths work from either the project root or notebooks folder.
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
SUBMISSION_DIR = PROJECT_DIR / "submissions"

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"


# ============================================================
# 2. LOAD DATA
# ============================================================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

assert TARGET in train.columns
assert TARGET not in test.columns
assert len(test) == len(sample_submission)


# ============================================================
# 3. CREATE FEATURES AND TARGET
# ============================================================

X = train.drop(columns=[TARGET, ID_COLUMN])
y = train[TARGET].astype(int)

X_test = test.drop(columns=[ID_COLUMN])

assert list(X.columns) == list(X_test.columns)

categorical_columns = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_columns = X.columns.difference(
    categorical_columns
).tolist()

print(f"\nTotal features:       {X.shape[1]}")
print(f"Numerical features:   {len(numeric_columns)}")
print(f"Categorical features: {len(categorical_columns)}")

print("\nNumerical columns:")
print(numeric_columns)

print("\nCategorical columns:")
print(categorical_columns)


# ============================================================
# 4. PREPROCESSING
# ============================================================

# Compatibility with older and newer scikit-learn versions.
try:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse_output=True
    )
except TypeError:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        sparse=True
    )


numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "one_hot",
            one_hot_encoder
        )
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_columns
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ],
    remainder="drop"
)


# ============================================================
# 5. MODEL
# ============================================================

model_parameters = {
    "C": 1.0,
    "solver": "lbfgs",
    "max_iter": 1000,
    "tol": 1e-5
}


# ============================================================
# 6. STRATIFIED CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

out_of_fold_predictions = np.zeros(
    len(train),
    dtype=float
)

test_predictions = np.zeros(
    len(test),
    dtype=float
)

fold_scores = []
fold_iterations = []

experiment_start = time()


for fold_number, (train_indices, validation_indices) in enumerate(
    cv.split(X, y),
    start=1
):
    fold_start = time()

    X_train_fold = X.iloc[train_indices]
    X_validation_fold = X.iloc[validation_indices]

    y_train_fold = y.iloc[train_indices]
    y_validation_fold = y.iloc[validation_indices]

    # Fit preprocessing only on the training portion of this fold.
    fold_preprocessor = clone(preprocessor)

    X_train_processed = fold_preprocessor.fit_transform(
        X_train_fold
    )

    X_validation_processed = fold_preprocessor.transform(
        X_validation_fold
    )

    X_test_processed = fold_preprocessor.transform(
        X_test
    )

    if fold_number == 1:
        print(
            "\nProcessed training matrix shape:",
            X_train_processed.shape
        )

    model = LogisticRegression(**model_parameters)

    model.fit(
        X_train_processed,
        y_train_fold
    )

    validation_probabilities = model.predict_proba(
        X_validation_processed
    )[:, 1]

    fold_test_probabilities = model.predict_proba(
        X_test_processed
    )[:, 1]

    out_of_fold_predictions[validation_indices] = (
        validation_probabilities
    )

    test_predictions += (
        fold_test_probabilities / N_SPLITS
    )

    fold_auc = roc_auc_score(
        y_validation_fold,
        validation_probabilities
    )

    fold_scores.append(fold_auc)

    iterations_used = int(model.n_iter_[0])
    fold_iterations.append(iterations_used)

    fold_minutes = (time() - fold_start) / 60

    print(
        f"Fold {fold_number}/{N_SPLITS} | "
        f"AUC: {fold_auc:.6f} | "
        f"Iterations: {iterations_used} | "
        f"Time: {fold_minutes:.2f} minutes"
    )


# ============================================================
# 7. VALIDATION RESULTS
# ============================================================

overall_oof_auc = roc_auc_score(
    y,
    out_of_fold_predictions
)

mean_fold_auc = float(np.mean(fold_scores))
std_fold_auc = float(np.std(fold_scores))

xgb_baseline_auc = 0.963034
difference_vs_xgb = overall_oof_auc - xgb_baseline_auc

total_minutes = (time() - experiment_start) / 60


print("\n" + "=" * 60)
print("EXPERIMENT 002 RESULTS")
print("=" * 60)

for fold_number, score in enumerate(
    fold_scores,
    start=1
):
    print(f"Fold {fold_number} AUC: {score:.6f}")

print(f"\nMean fold AUC:  {mean_fold_auc:.6f}")
print(f"Fold AUC SD:    {std_fold_auc:.6f}")
print(f"OOF AUC:        {overall_oof_auc:.6f}")
print(f"XGBoost OOF:    {xgb_baseline_auc:.6f}")
print(f"Difference:     {difference_vs_xgb:+.6f}")
print(f"Runtime:        {total_minutes:.2f} minutes")


# ============================================================
# 8. INTERPRET RESULT
# ============================================================

print("\nInterpretation:")

if overall_oof_auc > xgb_baseline_auc:
    print(
        "Logistic regression beat XGBoost. The dataset may be "
        "dominated by additive linear relationships."
    )

elif overall_oof_auc >= 0.960:
    print(
        "Logistic regression is extremely competitive. Most of "
        "the predictive signal appears to be approximately linear."
    )

elif overall_oof_auc >= 0.948:
    print(
        "The linear model is useful, but XGBoost gains meaningfully "
        "from nonlinear effects or feature interactions."
    )

else:
    print(
        "The linear model trails substantially. Nonlinear effects "
        "and interactions appear central to the problem."
    )


# ============================================================
# 9. PREDICTION SANITY CHECKS
# ============================================================

prediction_summary = pd.Series(
    test_predictions,
    name="predicted_probability"
).describe()

print("\nTest prediction summary:")
print(prediction_summary)

assert np.isfinite(test_predictions).all()

assert (
    (test_predictions >= 0) &
    (test_predictions <= 1)
).all()

assert np.std(test_predictions) > 0


# ============================================================
# 10. CONDITIONAL SUBMISSION
# ============================================================

if overall_oof_auc >= SUBMISSION_THRESHOLD:

    submission = sample_submission.copy()

    assert TARGET in submission.columns

    submission[TARGET] = test_predictions

    submission_path = (
        SUBMISSION_DIR /
        "exp_002_logistic_control.csv"
    )

    submission.to_csv(
        submission_path,
        index=False
    )

    print("\nSubmission created because OOF AUC met the threshold.")
    print(f"Saved to:\n{submission_path}")

    print("\nSubmission preview:")
    print(submission.head())

else:
    submission_path = None

    print(
        "\nNo submission created because OOF AUC was below "
        f"{SUBMISSION_THRESHOLD:.3f}."
    )

Train shape: (691369, 14)
Test shape:  (296302, 13)

Total features:       12
Numerical features:   9
Categorical features: 3

Numerical columns:
['age', 'app_opens_per_day', 'daily_screen_time_hours', 'gaming_hours', 'notifications_per_day', 'sleep_hours', 'social_media_hours', 'weekend_screen_time', 'work_study_hours']

Categorical columns:
['gender', 'stress_level', 'academic_work_impact']

Processed training matrix shape: (460912, 17)
Fold 1/3 | AUC: 0.910347 | Iterations: 11 | Time: 0.03 minutes
Fold 2/3 | AUC: 0.911866 | Iterations: 11 | Time: 0.03 minutes
Fold 3/3 | AUC: 0.912106 | Iterations: 11 | Time: 0.03 minutes

EXPERIMENT 002 RESULTS
Fold 1 AUC: 0.910347
Fold 2 AUC: 0.911866
Fold 3 AUC: 0.912106

Mean fold AUC:  0.911440
Fold AUC SD:    0.000779
OOF AUC:        0.911437
XGBoost OOF:    0.963034
Difference:     -0.051597
Runtime:        0.10 minutes

Interpretation:
The linear model trails substantially. Nonlinear effects and interactions appear central to the problem.

Te

### Experiment Log

In [2]:
from datetime import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd


EXPERIMENT_LOG_PATH = PROJECT_DIR / "experiment_log.csv"


def log_experiment(
    experiment_id,
    description,
    model,
    features,
    validation_method,
    cv_scores,
    kaggle_score=None,
    changes="",
    submission_file="",
    notes="",
    log_path=EXPERIMENT_LOG_PATH
):
    """
    Add or update one experiment in experiment_log.csv.

    If the experiment_id already exists, its previous row is replaced.
    """

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    cv_scores = [float(score) for score in cv_scores]

    cv_mean = float(np.mean(cv_scores))
    cv_std = float(np.std(cv_scores))

    kaggle_score_value = (
        float(kaggle_score)
        if kaggle_score is not None
        else np.nan
    )

    kaggle_cv_gap = (
        kaggle_score_value - cv_mean
        if pd.notna(kaggle_score_value)
        else np.nan
    )

    experiment_record = {
        "experiment_id": experiment_id,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "description": description,
        "model": model,
        "features": json.dumps(list(features)),
        "n_features": len(features),
        "validation_method": validation_method,
        "cv_scores": json.dumps(cv_scores),
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "kaggle_score": kaggle_score_value,
        "kaggle_cv_gap": kaggle_cv_gap,
        "changes": changes,
        "submission_file": submission_file,
        "notes": notes
    }

    if log_path.exists():
        experiments = pd.read_csv(log_path)

        # Prevent duplicate rows when rerunning the same experiment cell.
        if "experiment_id" in experiments.columns:
            experiments = experiments[
                experiments["experiment_id"] != experiment_id
            ].copy()
    else:
        experiments = pd.DataFrame()

    new_row = pd.DataFrame([experiment_record])

    experiments = pd.concat(
        [experiments, new_row],
        ignore_index=True
    )

    experiments = experiments.sort_values(
        by="experiment_id"
    ).reset_index(drop=True)

    experiments.to_csv(log_path, index=False)

    print(f"Logged {experiment_id}")
    print(f"CV mean:       {cv_mean:.6f}")
    print(f"CV SD:         {cv_std:.6f}")

    if pd.notna(kaggle_score_value):
        print(f"Kaggle score:  {kaggle_score_value:.6f}")
        print(f"Kaggle-CV gap: {kaggle_cv_gap:+.6f}")

    print(f"Log saved to:  {log_path}")

    return experiments

In [3]:
experiments = log_experiment(
    experiment_id="EXP-002",
    description=(
        "Logistic-regression control using standardized numerical features "
        "and most-frequent-imputed, one-hot-encoded categorical features."
    ),
    model="LogisticRegression",
    features=X.columns.tolist(),
    validation_method="3-fold StratifiedKFold with ROC AUC",
    cv_scores=[
        0.910347,
        0.911866,
        0.912106
    ],
    kaggle_score=None,
    changes=(
        "Replaced the XGBoost baseline with a regularized linear model. "
        "Added standardization for numerical features while retaining "
        "the same folds, features, imputation, and categorical encoding."
    ),
    submission_file="",
    notes=(
        "OOF AUC was 0.911437 with fold SD 0.000779. The model trailed "
        "the XGBoost baseline by 0.051597 AUC. No Kaggle submission was "
        "created because OOF AUC was below the predefined 0.960 threshold. "
        "The large gap indicates that nonlinear effects, thresholds, or "
        "feature interactions are important."
    )
)

experiments.tail()

Logged EXP-002
CV mean:       0.911440
CV SD:         0.000779
Log saved to:  C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\02-smartphone-addiction\experiment_log.csv


,experiment_id,timestamp,description,model,features,n_features,validation_method,cv_scores,cv_mean,cv_std,kaggle_score,kaggle_cv_gap,changes,submission_file,notes
0,EXP-001,2026-08-02 21:49:55,Initial XGBoost baseline using median-imputed ...,XGBClassifier,"[""age"", ""daily_screen_time_hours"", ""social_med...",12,3-fold StratifiedKFold with ROC AUC,"[0.962477, 0.963382, 0.963244]",0.963034,0.000398,0.96449,0.001456,Established the first end-to-end baseline usin...,exp_001_xgb_baseline.csv,OOF AUC was 0.963034 with fold SD 0.000398. Ka...
1,EXP-002,2026-08-02 21:53:06,Logistic-regression control using standardized...,LogisticRegression,"[""age"", ""daily_screen_time_hours"", ""social_med...",12,3-fold StratifiedKFold with ROC AUC,"[0.910347, 0.911866, 0.912106]",0.911440,0.000779,NaN,NaN,Replaced the XGBoost baseline with a regulariz...,,OOF AUC was 0.911437 with fold SD 0.000779. Th...
